#  Spark MLlib{background-color="black" background-image="https://blog.osservatori.net/hubfs/AI/machine-learning.jpg" background-size="80%" background-opacity="0.8"}

MLlib is Apache Spark's scalable machine learning library.


## Why MLlib in 2026?

:::: {.columns}

::: {.fragment .column width="50%"}
*It is still the right tool when...*

- data **does not fit** a single machine and is already in Spark (Kafka → Spark → Lakehouse)
- the model is **classical ML**: linear models, trees, clustering, ALS, frequent patterns
- you need **feature engineering at scale** before any model (also before an LLM!)
- training and scoring must live **inside the pipeline** (batch or Structured Streaming)
:::

::: {.fragment .column width="50%"}
*...and it is not when*

- the data fits in memory → scikit-learn / XGBoost / DuckDB are simpler
- you need **deep learning / pretrained language models** → Spark NLP, Hugging Face (next blocks)
- you need **LLM generation** → serve the model and call it (next lesson)

```python
data = spark.read.format("libsvm")\
  .load("hdfs://...")

model = KMeans(k=10).fit(data)
```
:::
::::

## Algorithms and Utilities
:::: {.columns}

::: {.fragment .column width="50%"}
*Algorithms*

* Classification: logistic regression, naive Bayes,...
* Regression: generalized linear regression, survival regression,...
* Decision trees, random forests, and gradient-boosted trees
* Recommendation: alternating least squares (ALS)
* Clustering: K-means, Gaussian mixtures (GMMs),...
* Topic modeling: latent Dirichlet allocation (LDA)
* Frequent itemsets, association rules, and sequential pattern mining
:::

::: {.fragment .column width="50%"}
*Utilities*

* Feature transformations: standardization, normalization, hashing,...
* ML Pipeline construction
* Model evaluation and hyper-parameter tuning
* ML persistence: saving and loading models and Pipelines
* Distributed linear algebra: SVD, PCA,...
* Statistics: summarizer, correlation, hypothesis testing
:::
::::

## [DataFrame-based API (`spark.ml`) is the API](https://spark.apache.org/docs/latest/ml-guide.html)

:::: {.columns}

::: {.fragment .column width="50%"}
- `pyspark.ml` → **DataFrame-based**, the one to use
- `pyspark.mllib` → **RDD-based**, maintenance mode since Spark 2.0, no new features
- "Spark ML" is not an official name: it just refers to the DataFrame API
:::

::: {.fragment .column width="50%"}
*What changed in Spark 4.x*

- `pyspark.ml` works over **Spark Connect** (thin client, remote cluster)
- Python **3.9+**, Java **17+**, Scala **2.13**
- `findspark` is not needed anymore: `pip install pyspark` is a full local Spark
:::
::::

## Data types: vectors

- Features are stored in a single column of type `Vector` (dense or sparse)
- `(1.0, 0.0, 3.0)` is
  - dense → `[1.0, 0.0, 3.0]`
  - sparse → `(3, [0, 2], [1.0, 3.0])` = (size, indices, values)
- Text features (TF, TF-IDF) are almost always **sparse**
- `VectorAssembler` builds the features column from many columns

### Example vector

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors

spark = SparkSession.builder.appName("MLlib essentials").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

dense = Vectors.dense([1.0, 0.0, 3.0])
sparse = Vectors.sparse(3, [0, 2], [1.0, 3.0])
print(dense, sparse, "same values:", dense.toArray().tolist() == sparse.toArray().tolist())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/17 20:37:36 WARN Utils: Your hostname, NicSBook, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/17 20:37:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 20:37:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[1.0,0.0,3.0] (3,[0,2],[1.0,3.0]) same values: True


## The three abstractions you must know

:::: {.columns}

::: {.fragment .column width="33%"}
**Transformer**

`df → df`

`transform()` adds a column.

e.g. `Tokenizer`, `HashingTF`, a fitted model
:::

::: {.fragment .column width="33%"}
**Estimator**

`df → Transformer`

`fit()` learns parameters from data.

e.g. `IDF`, `StringIndexer`, `LogisticRegression`
:::

::: {.fragment .column width="33%"}
**Pipeline**

A chain of stages; it is itself an **Estimator**.

`fit()` returns a `PipelineModel` (a Transformer) that you can **save** and **reload** in a streaming job.
:::
::::

::: {.fragment}
![](https://spark.apache.org/docs/latest/img/ml-Pipeline.png)
:::

::: {.fragment}
Spark NLP (next block) reuses **exactly these abstractions**: an NLP annotator is a Transformer or an Estimator.
:::

## Tweet sentiment

Sentiment140 (1.6M tweets, labels 0/4) — the same dataset of [Mlib - Sentiment Analysis (English)](Mlib%20-%20Sentiment%20Analysis%20%28English%29.ipynb)

### Load data

In [2]:
from pyspark.sql import functions as F

schema = "target short, id long, ts string, flag string, user string, text string"
tweets = (spark.read.csv("../spark/dataset/training.1600000.processed.noemoticon.csv.gz",
                         schema=schema, encoding="ISO-8859-1")
          .withColumn("label", F.when(F.col("target") == 4, 1.0).otherwise(0.0))
          .select("text", "label")
          .sample(fraction=0.1, seed=42))          # 10% is enough in class

train, test = tweets.randomSplit([0.8, 0.2], seed=42)
train.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|63974|
|  1.0|64099|
+-----+-----+



### Create the pipeline

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression

pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+"),   # Transformer
    StopWordsRemover(inputCol="tokens", outputCol="words"),                # Transformer
    HashingTF(inputCol="words", outputCol="tf", numFeatures=2**16),        # Transformer
    IDF(inputCol="tf", outputCol="features", minDocFreq=5),               # Estimator
    LogisticRegression(maxIter=50),                                       # Estimator
])

model = pipeline.fit(train)          # PipelineModel

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


### Evaluate, persist, reuse

In [4]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

pred = model.transform(test)
print("AUC     ", BinaryClassificationEvaluator().evaluate(pred))
print("accuracy", MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred))

model.write().overwrite().save("/tmp/tap-sentiment-tfidf")   # reload it in a streaming job

AUC      0.7982897392077959


accuracy 0.7386716325263811


### Test the model on a few tweets

In [5]:
from pyspark.ml import PipelineModel

reloaded = PipelineModel.load("/tmp/tap-sentiment-tfidf")
samples = spark.createDataFrame([("I love Spark",), ("This lesson is not boring at all",),
                                 ("I hate waiting for the bus",)], ["text"])
reloaded.transform(samples).select("text", "prediction", "probability").show(truncate=False)

+--------------------------------+----------+----------------------------------------+
|text                            |prediction|probability                             |
+--------------------------------+----------+----------------------------------------+
|I love Spark                    |1.0       |[0.20751837235131945,0.7924816276486806]|
|This lesson is not boring at all|0.0       |[0.7097898581263418,0.29021014187365823]|
|I hate waiting for the bus      |0.0       |[0.8837516974846936,0.11624830251530638]|
+--------------------------------+----------+----------------------------------------+



## Where bag-of-words breaks

- ***"This lesson is not boring at all"*** → `not`, `boring` ... a TF-IDF model sees *boring*
- No word order, no context, no knowledge of the language: everything is learned from **our** labels
- On our 10% sample: **AUC ≈ 0.80, accuracy ≈ 0.74** — and *"not boring at all"* is classified **negative**
- Every new language or task needs new labelled data

::: {.fragment}
*Pretrained transformers* (BERT, RoBERTa, ...) learn language once and are **fine-tuned** or used **zero-shot**: they routinely reach 90%+ on the same task.

How do we run them on Spark? → **[Spark NLP](SparkNLP.ipynb)** and **[Hugging Face in Spark](Spark%20Hugging%20Face.ipynb)**
:::

## Self-study notebooks

The detailed MLlib material is available as runnable notebooks (Spark 4.x, local `pyspark`):

| Notebook | What you will find | Effort |
|---|---|---|
| [Mlib - Basic Statistics](Mlib%20-%20Basic%20Statistics.ipynb) | Summarizer, skewness, Pearson/Spearman correlation, Chi-square test (Zoo dataset) | 30' |
| [Mlib - Pipelines](Mlib%20-%20Pipelines.ipynb) | Transformers, Estimators, Pipelines, persistence | 20' |
| [MLib - ETS Features](MLib%20-%20ETS%20Features.ipynb) | Feature extraction, transformation, selection, LSH (reference catalogue) | reference |
| [MLib - Data Sources - Classification and regression - Clustering](MLib%20-%20Data%20Sources%20-%20Classification%20and%20regression%20-%20Clustering.ipynb) | LIBSVM, logistic regression, ROC/F-measure, K-means | 40' |
| [Mlib - Sentiment Analysis (English)](Mlib%20-%20Sentiment%20Analysis%20%28English%29.ipynb) | Word2Vec, Naive Bayes, TF-IDF, Random Forest on Sentiment140 | 60' |

::: {.fragment}
**Challenge**: compare the TF-IDF model above with the Spark NLP RoBERTa model on the same 1,000 test tweets. Report accuracy *and* time per 1,000 rows.
:::

### Stop lesson

In [6]:
spark.stop()

# Biblio

* https://spark.apache.org/mllib/
* https://spark.apache.org/docs/latest/ml-guide.html
* https://spark.apache.org/docs/latest/ml-pipeline.html
* https://spark.apache.org/docs/latest/ml-features.html
* https://spark.apache.org/docs/latest/spark-connect-overview.html
* Adi Polak, *Scaling Machine Learning with Spark*, O'Reilly 2023
* https://runawayhorse001.github.io/LearningApacheSpark/index.html